In [30]:
import pandas as pd
import numpy as np
import pandas_market_calendars as mcal
import importlib
import config
import sys
sys.path.insert(0, '../scripts')
from scripts.remove_non_trading_days import remove_non_trading_days
importlib.reload(config)
import tabulate

from config import BB_PRICES_COMPLETE, US_TICKERS_FINAL_adj,PRICE_FEATURES

In [2]:
pdf = pd.read_parquet(BB_PRICES_COMPLETE)

In [3]:
#Get the sector from US Tickers file and merge it into the pdf.
ust = pd.read_csv(US_TICKERS_FINAL_adj)

#Create a sector map to make sure there are no duplicates. Then merge on that map.
sec_map = ust[['ISIN', 'Sector']].drop_duplicates()
assert not sec_map['ISIN'].duplicated().any(), \
    "Same ISIN maps to two Sectors - resolve before merging"

n_before = len(pdf)
pdf = pdf.merge(sec_map, on='ISIN', how='left')
assert len(pdf) == n_before, "Merge changed row count!"

print(f"Rows without sector: {pdf['Sector'].isna().sum():,}")

Rows without sector: 0


In [4]:
#Run this to make sure there are no non-trading days rows in your df
before = len(pdf.index)
pdf = remove_non_trading_days(pdf,'date')
after = len(pdf.index)
print(f'lines before clean: {before}, lines after clean: {after}, therefore {before-after} lines removed')

There are currently 1921 days in your df
There are 1921 days in the NYSE schedule
Therefore we remove 0 days from the df
df shape after cleaning:(2408260, 6)
lines before clean: 2408260, lines after clean: 2408260, therefore 0 lines removed


In [5]:
pdf.columns

Index(['ISIN', 'Ticker', 'resolved_ticker', 'date', 'px_last', 'Sector'], dtype='object')

In [6]:
pdf = pdf.sort_values(['ISIN', 'date']).reset_index(drop=True)
pdf.rename(columns={'resolved_ticker': 'bb_tcm'}, inplace=True)

In [7]:
#Before constructing the features assert some conditions:
# 1. No weekends (Mon=0 ... Sun=6)
assert pdf['date'].dt.dayofweek.max() <= 4, "Weekend rows present!"

# 2. No duplicate (ISIN, day) pairs - shifts silently misbehave if there are
assert not pdf.duplicated(['ISIN', 'date']).any(), "Duplicate rows!"

# 3. No missing or non-positive prices - log() would produce -inf/NaN silently
assert (pdf['px_last'] > 0).all(), "Zero/negative/missing prices!"

In [8]:
# Log price once - every return is then a subtraction
pdf['log_p'] = np.log(pdf['px_last'])

# Group object we'll reuse throughout
g = pdf.groupby('ISIN')['log_p']

pdf['r0']     = pdf['log_p'] - g.shift(1)     # ln(P0)  - ln(P-1)
pdf['r_1']    = g.shift(1)   - g.shift(2)     # ln(P-1) - ln(P-2)
pdf['r_5_2']  = g.shift(2)   - g.shift(5)     # ln(P-2) - ln(P-5)
pdf['r_10_6'] = g.shift(5)   - g.shift(10)    # ln(P-5) - ln(P-10)


In [9]:
#Automated check to see if what's above is doing what was intended.
tile_sum = pdf[['r0', 'r_1', 'r_5_2', 'r_10_6']].sum(axis=1, min_count=4)
ten_day  = pdf['log_p'] - g.shift(10)
assert np.allclose(tile_sum.dropna(), ten_day.dropna()), "Tiling broken!"

In [10]:
#Manaual check to see if what's above is doing what was intended.

rng = np.random.default_rng(42)
isin = rng.choice(pdf['ISIN'].unique())
sub = pdf[pdf['ISIN'] == isin].reset_index(drop=True)

start = rng.integers(10, len(sub) - 20)  # start >= 10 so the lookbacks are populated
sample = sub.loc[start:start+19,
                 ['ISIN', 'date', 'px_last','log_p', 'r0', 'r_1', 'r_5_2', 'r_10_6']]
print(sample.to_string())

              ISIN       date  px_last     log_p        r0       r_1     r_5_2    r_10_6
1471  US0530151036 2023-10-12   247.65  5.512016 -0.007282  0.001243  0.023513  0.000205
1472  US0530151036 2023-10-13   247.50  5.511411 -0.000606 -0.007282  0.012424  0.023822
1473  US0530151036 2023-10-16   249.26  5.518497  0.007086 -0.000606 -0.010164  0.044170
1474  US0530151036 2023-10-17   249.33  5.518777  0.000281  0.007086 -0.006645  0.035668
1475  US0530151036 2023-10-18   248.26  5.514477 -0.004301  0.000281 -0.000802  0.022212
1476  US0530151036 2023-10-19   246.08  5.505657 -0.008820 -0.004301  0.006761  0.017475
1477  US0530151036 2023-10-20   241.68  5.487615 -0.018042 -0.008820  0.003066  0.004536
1478  US0530151036 2023-10-23   241.16  5.485461 -0.002154 -0.018042 -0.012840 -0.003684
1479  US0530151036 2023-10-24   240.45  5.482512 -0.002948 -0.002154 -0.031163  0.000722
1480  US0530151036 2023-10-25   218.33  5.386008 -0.096504 -0.002948 -0.029016 -0.004822
1481  US0530151036 20

In [11]:
# 20-day moving average of price, per stock
pdf['ma20'] = pdf.groupby('ISIN')['px_last'].transform(
    lambda s: s.rolling(20).mean()
)

# Log deviation of price from its 20-day average
#pvma = price vs moving average
pdf['pvma'] = np.log(pdf['px_last'] / pdf['ma20'])

In [12]:
# Baseline = the 20 daily returns ending YESTERDAY (days -20 to -1)
r0_lag = pdf.groupby('ISIN')['r0'].shift(1)          # yesterday's r0, per stock

mu  = r0_lag.groupby(pdf['ISIN']).transform(lambda s: s.rolling(20).mean())
sig = r0_lag.groupby(pdf['ISIN']).transform(lambda s: s.rolling(20).std())

#r0_z20 = z score of log return today vs previous 20 days (today's return not included in the calc)
# produce absurd z-scores from a near-zero denominator
sig = sig.clip(lower=1e-6)
pdf['r0_z20'] = (pdf['r0'] - mu) / sig

# Winsorise at +/-5, consistent with the sentiment z-scores
pdf['r0_z20'] = pdf['r0_z20'].clip(-5, 5)

In [13]:
print(pdf[['pvma', 'r0_z20']].describe())

               pvma        r0_z20
count  2.380349e+06  2.377411e+06
mean  -3.233948e-04 -1.821393e-02
std    6.867615e-02  1.134118e+00
min   -6.630445e+00 -5.000000e+00
25%   -2.813683e-02 -6.251195e-01
50%    3.648540e-03 -1.194161e-03
75%    3.260496e-02  6.193690e-01
max    1.888334e+00  5.000000e+00


In [47]:
worst = pdf.nsmallest(10, 'pvma')[['ISIN','bb_tcm' ,'date', 'px_last', 'ma20', 'pvma']]
print(worst.to_string())

                 ISIN   bb_tcm       date  px_last       ma20      pvma
1969758  US82669G1040  SBNY US 2023-03-28  0.13000  98.516500 -6.630445
1969760  US82669G1040  SBNY US 2023-03-30  0.18875  85.971438 -6.121347
1969761  US82669G1040  SBNY US 2023-03-31  0.18300  80.141587 -6.082064
1969762  US82669G1040  SBNY US 2023-04-03  0.16990  74.301083 -6.080671
1969763  US82669G1040  SBNY US 2023-04-04  0.16800  68.495483 -6.010559
1969764  US82669G1040  SBNY US 2023-04-05  0.15870  62.792918 -5.980582
1969759  US82669G1040  SBNY US 2023-03-29  0.24000  92.177500 -5.950832
1969765  US82669G1040  SBNY US 2023-04-06  0.16500  57.110667 -5.846801
1969766  US82669G1040  SBNY US 2023-04-10  0.15000  51.365667 -5.836090
1969767  US82669G1040  SBNY US 2023-04-11  0.14200  45.742268 -5.774951


In [48]:
print(pdf['r0'].eq(0).mean())               # was worth checking at ~? before; expect a drop
print(pdf['r0_z20'].abs().eq(5).mean())     # winsorisation rate
print(pdf[['ma20','r0_z20']].isna().sum())  # warm-up NaNs only, ~21 x 1,469

0.007551510218996288
0.004655643493642713
ma20      27911
r0_z20    30849
dtype: int64


In [15]:
MIN_GROUP_SIZE = 4  # stock itself + at least 3 peers with news that day
STD_FLOOR = 1e-6 #prevents floating-point ghost variance
Z_CAP = 5 #winsorisation bound

def pricing_add_loo_zscore(df, value_col, z_col, min_group_size):
    """
    Adds a leave-one-out z-score column: how unusual is this stock's value
    relative to the OTHER covered stocks in the same (Sector, trading_day)
    group. Groups smaller than min_group_size get z = 0.
    """
    grp = df.groupby(['Sector', 'date'], sort=False)

    x = df[value_col]
    n = grp[value_col].transform('count')
    s = grp[value_col].transform('sum')

    df['_sq'] = x ** 2
    q = df.groupby(['Sector', 'date'], sort=False)['_sq'].transform('sum')
    df.drop(columns='_sq', inplace=True)

    # Mean of the group excluding the stock itself
    loo_mean = (s - x) / (n - 1)

    # Sample variance (ddof=1) of the group excluding the stock itself:
    #   peers' sum of squares  = q - x^2
    #   peers' count           = n - 1
    #   var = (sum_sq - count * mean^2) / (count - 1)

    loo_var = (q - x ** 2 - (n - 1) * loo_mean ** 2) / (n - 2)
    loo_std = np.sqrt(loo_var.clip(lower=0.0))

    valid = (n >= min_group_size) & (loo_std > STD_FLOOR)

    z = np.where(valid, (x - loo_mean) / loo_std.where(valid, 1.0), np.nan)
    df[z_col] = np.clip(z, -Z_CAP, Z_CAP)

    return df

In [16]:
#Create the performance z-score relative to it's sector peers.

#Create the temporary 20 day return of the stock.
pdf['r_20'] = pdf['log_p'] - pdf.groupby('ISIN')['log_p'].shift(20)

pricing_add_loo_zscore(pdf, 'r_20', 'Sector_z20', min_group_size=MIN_GROUP_SIZE)



,ISIN,Ticker,bb_tcm,date,px_last,Sector,log_p,r0,r_1,r_5_2,r_10_6,ma20,pvma,r0_z20,r_20,Sector_z20
0,US0003602069,AAON,AAON US,2017-12-05,25.033333,Industrials,3.220208,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,US0003602069,AAON,AAON US,2017-12-06,24.200000,Industrials,3.186353,-0.033856,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,US0003602069,AAON,AAON US,2017-12-07,24.166667,Industrials,3.184974,-0.001378,-0.033856,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,US0003602069,AAON,AAON US,2017-12-08,23.800000,Industrials,3.169686,-0.015289,-0.001378,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,US0003602069,AAON,AAON US,2017-12-11,23.366667,Industrials,3.151311,-0.018375,-0.015289,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2408255,US98980F1049,ZI,GTM US,2025-07-08,10.150000,Communication Services,2.317474,0.010897,-0.021676,0.013739,0.040328,9.9830,0.016590,0.520344,0.023929,-0.326062
2408256,US98980F1049,ZI,GTM US,2025-07-09,10.430000,Communication Services,2.344686,0.027213,0.010897,-0.003976,0.003976,9.9970,0.042401,1.278962,0.027213,-0.307693
2408257,US98980F1049,ZI,GTM US,2025-07-10,10.370000,Communication Services,2.338917,-0.005769,0.027213,0.010897,0.030336,10.0095,0.035382,-0.347005,0.024403,-0.290472
2408258,US98980F1049,ZI,GTM US,2025-07-11,10.200000,Communication Services,2.322388,-0.016529,-0.005769,0.016433,0.033700,10.0075,0.019053,-0.862146,-0.003914,-0.466439


In [21]:
#Create the forward returns 1, 3 and 5 day. and then the same vs sector peers' performance.
g = pdf.groupby('ISIN')['log_p']

pdf['fwd_r1'] = g.shift(-1) - pdf['log_p']
pdf['fwd_r3'] = g.shift(-3) - pdf['log_p']
pdf['fwd_r5'] = g.shift(-5) - pdf['log_p']

def add_loo_demean(df, value_col, out_col, min_group_size):
    """LOO sector demean: value minus the mean of same-(Sector, date)
    peers, excluding the stock itself. Mutates df in place; returns None."""
    grp = df.groupby(['Sector', 'date'], sort=False)
    x = df[value_col]
    n = grp[value_col].transform('count')
    s = grp[value_col].transform('sum')

    loo_mean = (s - x) / (n - 1)
    valid = n >= min_group_size
    df[out_col] = np.where(valid, x - loo_mean, np.nan)

for k in (1, 3, 5):
    add_loo_demean(pdf, f'fwd_r{k}', f'fwd_r{k}_sec', MIN_GROUP_SIZE)

In [33]:
#Save the features nicely structured
FEATURES = ['r0', 'r_1', 'r_5_2', 'r_10_6', 'pvma', 'r0_z20', 'Sector_z20']
TARGETS  = ['fwd_r1', 'fwd_r3', 'fwd_r5',
            'fwd_r1_sec', 'fwd_r3_sec', 'fwd_r5_sec']
KEYS     = ['ISIN','bb_tcm', 'date', 'Sector']

pdf[KEYS + FEATURES + TARGETS].to_parquet(PRICE_FEATURES, index=False)

In [32]:
print(pdf[TARGETS].describe().to_markdown())

|       |       fwd_r1 |       fwd_r3 |       fwd_r5 |   fwd_r1_sec |   fwd_r3_sec |   fwd_r5_sec |
|:------|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|
| count |  2.40679e+06 |  2.40385e+06 |  2.40092e+06 |  2.40678e+06 |  2.40384e+06 |  2.40091e+06 |
| mean  |  0.000106456 |  0.000310142 |  0.000509311 | -1.24548e-21 | -1.99521e-20 |  1.73869e-20 |
| std   |  0.0276095   |  0.0473081   |  0.0606162   |  0.0223825   |  0.0383725   |  0.0490669   |
| min   | -6.28872     | -6.28872     | -6.54844     | -6.30308     | -6.30397     | -6.48379     |
| 25%   | -0.0108584   | -0.019091    | -0.0247756   | -0.00842671  | -0.0150653   | -0.0197804   |
| 50%   |  0.00051386  |  0.00144474  |  0.0023307   |  0.000128427 |  0.000259588 |  0.000405832 |
| 75%   |  0.0115431   |  0.0210633   |  0.0279674   |  0.00860702  |  0.0155277   |  0.020562    |
| max   |  1.70181     |  1.91316     |  2.96294     |  1.70368     |  1.85575     |  2.91571     |


In [34]:

print(pdf[TARGETS].isna().sum())    # expect ~k x 1,469 per raw target, more for _sec
# spot-check one stock's tail: last 5 rows should show fwd_r5 = NaN
print(pdf[pdf['ISIN'] == 'US82669G1040'][['date','fwd_r1','fwd_r5']].tail(7))

fwd_r1        1469
fwd_r3        4407
fwd_r5        7345
fwd_r1_sec    1478
fwd_r3_sec    4416
fwd_r5_sec    7354
dtype: int64
              date    fwd_r1    fwd_r5
1970251 2025-03-17 -0.084083  0.072571
1970252 2025-03-18  0.115832  0.156654
1970253 2025-03-19 -0.010471       NaN
1970254 2025-03-20  0.051293       NaN
1970255 2025-03-21  0.000000       NaN
1970256 2025-03-24  0.000000       NaN
1970257 2025-03-25       NaN       NaN
